1: clone and re-create the data (in a subprocess, before installing Unsloth)

In [1]:
!git clone --depth 1 -b main https://github.com/FatiAllali123/SLM-Software-Engineering.git /kaggle/working/SLM-Software-Engineering
!cp /kaggle/working/SLM-Software-Engineering/finetuning/prepare_data.py /kaggle/working/prepare_data.py
!python /kaggle/working/prepare_data.py 2>&1 | grep -E "removed total|python pool|train token stats|wrote"

Cloning into '/kaggle/working/SLM-Software-Engineering'...
remote: Enumerating objects: 72, done.
remote: Counting objects: 100% (72/72), done.
remote: Compressing objects: 100% (62/62), done.
remote: Total 72 (delta 23), reused 30 (delta 8), pack-reused 0 (from 0)
Receiving objects: 100% (72/72), 160.66 KiB | 2.59 MiB/s, done.
Resolving deltas: 100% (23/23), done.
python pool: 38284
removed total 3562 (n-gram 3326, name 361, both 125) -> kept 34722
train token stats: {'mean': 523.5, 'median': 508, 'max': 1698} | too long dropped: 1 of 6301
wrote /kaggle/working/ft_data


2: verify the fingerprint


In [2]:
import hashlib
def sha(p): return hashlib.sha256(open(p, "rb").read()).hexdigest()
EXPECTED = {
  "train.jsonl": "8b2d6e743b8f8a23b89d20f1cc0f97d823a522134fe291e5aab0655dc7df2b2a",
  "val.jsonl":   "01b59b3f8b4ad639993e20f566038d3942c3c55f291e5f4d4fbbff404e3cc116",
}
for f, h in EXPECTED.items():
    got = sha(f"/kaggle/working/ft_data/{f}")
    print(f, "OK" if got == h else f"MISMATCH {got}")
    assert got == h, "Data differs from the prep run: stop and investigate"

train.jsonl OK
val.jsonl OK


3: install the pinned versions and record the environment

In [3]:
!pip install -q unsloth==2026.11.3 unsloth_zoo==2026.11.3 trl==1.15.0
!pip freeze > /kaggle/working/pip_freeze_finetune.txt
!grep -iE "^(unsloth|unsloth_zoo|trl|transformers|peft|torch|bitsandbytes|datasets|accelerate|xformers)==" /kaggle/working/pip_freeze_finetune.txt

accelerate==1.15.0
bitsandbytes==0.50.2
datasets==4.8.5
peft==0.21.2
torch==2.11.0+cu130
transformers==5.17.0
trl==1.15.0
unsloth==2026.11.3
unsloth_zoo==2026.11.3
xformers==0.0.35


4: model, data, trainer

In [4]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
from unsloth import FastLanguageModel, is_bfloat16_supported
from unsloth.chat_templates import train_on_responses_only
import torch, json, time
from datasets import Dataset
from trl import SFTTrainer, SFTConfig

CFG = {"model": "Qwen/Qwen2.5-Coder-1.5B-Instruct", "max_seq_length": 2048, "r": 16, "lora_alpha": 16,
       "lora_dropout": 0, "lr": 2e-4, "per_device_batch": 2, "grad_accum": 8, "seed": 42,
       "epochs": 1, "warmup_steps": 20, "eval_steps": 125, "save_steps": 100, "run_name": "run1"}

model, tok = FastLanguageModel.from_pretrained(CFG["model"], max_seq_length=CFG["max_seq_length"],
                                               load_in_4bit=True, dtype=None)
model = FastLanguageModel.get_peft_model(
    model, r=CFG["r"], lora_alpha=CFG["lora_alpha"], lora_dropout=CFG["lora_dropout"],
    target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
    use_gradient_checkpointing="unsloth", random_state=CFG["seed"])

def load(path):
    rows = [json.loads(l) for l in open(path, encoding="utf-8")]
    return Dataset.from_list([{"text": tok.apply_chat_template(r["messages"], tokenize=False)} for r in rows])
train_ds, val_ds = load("/kaggle/working/ft_data/train.jsonl"), load("/kaggle/working/ft_data/val.jsonl")
assert train_ds[0]["text"].startswith("<|im_start|>system\nYou are Qwen")

trainer = SFTTrainer(
    model=model, processing_class=tok, train_dataset=train_ds, eval_dataset=val_ds,
    args=SFTConfig(
        output_dir=f"/kaggle/working/out_{CFG['run_name']}",
        per_device_train_batch_size=CFG["per_device_batch"], gradient_accumulation_steps=CFG["grad_accum"],
        per_device_eval_batch_size=2, num_train_epochs=CFG["epochs"], warmup_steps=CFG["warmup_steps"],
        learning_rate=CFG["lr"], lr_scheduler_type="linear", optim="adamw_8bit", weight_decay=0.01,
        fp16=not is_bfloat16_supported(), bf16=is_bfloat16_supported(),
        logging_steps=5, eval_strategy="steps", eval_steps=CFG["eval_steps"],
        save_strategy="steps", save_steps=CFG["save_steps"], save_total_limit=2,
        seed=CFG["seed"], report_to="none", max_length=CFG["max_seq_length"],
        dataset_text_field="text", packing=False, dataset_num_proc=1))
trainer = train_on_responses_only(trainer, instruction_part="<|im_start|>user\n",
                                  response_part="<|im_start|>assistant\n")
ex = trainer.train_dataset[0]
print("last tokens:", repr(tok.decode(ex["input_ids"][-8:])))
lab = [t for t, l in zip(ex["input_ids"], ex["labels"]) if l != -100]
print("trained-on starts:", repr(tok.decode(lab[:40])), "|", len(lab), "of", len(ex["input_ids"]))

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
Unsloth: xformers is installed but cannot load with torch 2.11.0+cu130 (it was built for 2.10.0+cu128), so PyTorch attention is used instead.
To install a matching xformers, run `unsloth install-kernels xformers`, or `pip uninstall xformers` to hide this. UNSLOTH_ENABLE_LOGGING=1 shows the full error.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.11.3: Fast Qwen2 patching. Transformers: 5.17.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.562 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu130. CUDA: 7.5. CUDA Toolkit: 13.0. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


/usr/local/lib/python3.13/dist-packages/huggingface_hub/constants.py:302: FutureWarning: The `HF_HUB_ENABLE_HF_TRANSFER` environment variable is deprecated as 'hf_transfer' is not used anymore. Please use `HF_XET_HIGH_PERFORMANCE` instead to enable high performance transfer with Xet. Visit https://huggingface.co/docs/huggingface_hub/package_reference/environment_variables#hfxethighperformance for more details.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/huggingface_hub/constants.py:302: FutureWarning: The `HF_HUB_ENABLE_HF_TRANSFER` environment variable is deprecated as 'hf_transfer' is not used anymore. Please use `HF_XET_HIGH_PERFORMANCE` instead to enable high performance transfer with Xet. Visit https://huggingface.co/docs/huggingface_hub/package_reference/environment_variables#hfxethighperformance for more details.
  warnings.warn(


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Unsloth 2026.11.3 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


Unsloth: compiling each transformer block as one region (Qwen2DecoderLayer).


Unsloth: Tokenizing ["text"]:   0%|          | 0/6000 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"]:   0%|          | 0/300 [00:00<?, ? examples/s]

🦥 Unsloth: Padding-free auto-enabled, enabling faster training.


Map (num_proc=2):   0%|          | 0/6000 [00:00<?, ? examples/s]

Map (num_proc=2):   0%|          | 0/300 [00:00<?, ? examples/s]

last tokens: " 'update_xylim'}\n```<|im_end|>\n"
trained-on starts: '```python\nimport ast\n\ndef extract_info(code_snippets):\n    result = {"wcropy_max": None, "observe_function": None}\n    \n    for snippet in code_snippets:\n        try' | 299 of 608


Check 2 is what tells us the loss is computed on the answer only. If the trained-on text begins with the problem statement instead of the code block, tell me before training.

5: train

In [5]:
import statistics
before = trainer.evaluate()["eval_loss"]; print("val loss before (300 ex):", before)
torch.cuda.reset_peak_memory_stats(); t0 = time.time()
out = trainer.train()
wall = time.time() - t0
after = trainer.evaluate()["eval_loss"]; print("val loss after (300 ex):", after)
print(f"train wall {wall/60:.1f} min | peak GPU mem {torch.cuda.max_memory_allocated()/1e9:.2f} GB")

Filter:   0%|          | 0/300 [00:00<?, ? examples/s]

Unsloth: Not an error, but `num_items_in_batch` could not be counted for Qwen2ForCausalLM's labels, so each micro-batch loss is a mean.
Using gradient accumulation will be very slightly less accurate.
Read more on gradient accumulation issues here: https://unsloth.ai/blog/gradient


Training Loss,Validation Loss,Step
No log,0.346472,0


The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 6,000 | Num Epochs = 1 | Total steps = 375
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 8 x 1) = 16
 "-____-"     Trainable parameters = 18,464,768 of 1,562,179,072 (1.18% trained)


val loss before (300 ex): 0.3464721739292145
Unsloth: Will smartly offload gradients to save VRAM!
Unsloth: packed / padding-free rows run each transformer block uncompiled; only unpacked batches use the compiled blocks.
Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Step,Training Loss,Validation Loss
125,0.324942,0.313624
250,0.284826,0.308138
375,0.286451,0.305957


Unsloth: Restored added_tokens_decoder metadata in /kaggle/working/out_run1/checkpoint-100/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in /kaggle/working/out_run1/checkpoint-200/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in /kaggle/working/out_run1/checkpoint-300/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in /kaggle/working/out_run1/checkpoint-375/tokenizer_config.json.


Training Loss,Validation Loss,Step
0.286451,0.305957,375


val loss after (300 ex): 0.30595678091049194
train wall 41.4 min | peak GPU mem 1.77 GB


6: save the adapter and a report

In [6]:
from importlib.metadata import version
import shutil
ADIR = f"/kaggle/working/adapter_{CFG['run_name']}"
model.save_pretrained(ADIR); tok.save_pretrained(ADIR)
rep = {"cfg": CFG, "gpu": torch.cuda.get_device_name(0), "val_loss_before": before, "val_loss_after": after,
       "train_runtime_s": out.metrics["train_runtime"], "total_steps": trainer.state.global_step,
       "peak_gpu_gb": round(torch.cuda.max_memory_allocated()/1e9, 2),
       "log_history": trainer.state.log_history,
       "versions": {k: version(k) for k in ["unsloth","unsloth_zoo","trl","transformers","peft","torch","bitsandbytes","datasets"]},
       "data_sha256": EXPECTED}
json.dump(rep, open(f"/kaggle/working/{CFG['run_name']}_report.json", "w"), indent=2)
shutil.make_archive(ADIR, "zip", ADIR)
print("saved:", ADIR, "and", f"{CFG['run_name']}_report.json")

Unsloth: Restored added_tokens_decoder metadata in /kaggle/working/adapter_run1/tokenizer_config.json.


saved: /kaggle/working/adapter_run1 and run1_report.json
